# One API call

Exercise: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07a-llm-api-call-exercise.ipynb) Solution: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07a-llm-api-call.ipynb)

A call to the Responses API sends two strings. `instructions` sets the role the model plays.
`input` carries the question. The reply comes back in `output_text`. The `usage` field counts
the tokens read and written, and tokens are what you pay for.

In [1]:
!pip install -q openai pypdf chromadb tiktoken python-dotenv
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()                                   # local: reads .env in this folder if present
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata       # Colab: Settings > Secrets > OPENAI_API_KEY
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
# A key bound to a region fails with 401 "incorrect regional hostname"; then also set
# OPENAI_BASE_URL, for example https://us.api.openai.com/v1 (Colab secret or .env).
client = OpenAI()
MODEL = "gpt-5.6-luna"   # $0.20 in / $1.20 out per 1M tokens, developers.openai.com/api/docs/pricing, 2026-09-10

In [2]:
resp = client.responses.create(
    model=MODEL,
    instructions="You are a geotechnical engineer.",
    input="Estimate the ultimate bearing capacity of a 2 m diameter circular footing at the ground surface on clay with undrained shear strength 35 kPa. Show the formula and the factors you use.",
)
print(resp.output_text)
print(f"tokens in={resp.usage.input_tokens} out={resp.usage.output_tokens}")

Assume **short-term undrained loading** of homogeneous clay:

- Undrained shear strength: \(c_u = 35\ \text{kPa}\)
- Undrained friction angle: \(\phi_u = 0^\circ\)
- Circular footing diameter: \(D=2\ \text{m}\)
- Embedment: \(D_f=0\), so surcharge \(q=\gamma D_f=0\)
- Vertical, concentric loading; no inclination or eccentricity effects
- No factor of safety applied

Using the general bearing-capacity equation for undrained clay:

\[
q_{ult}=c_u N_c s_c d_c + q
\]

For \(\phi=0^\circ\):

\[
N_c=5.14
\]

For a circular footing, use the shape factor:

\[
s_c=1.2
\]

At the ground surface:

\[
q=0,\qquad d_c=1
\]

Therefore,

\[
q_{ult}=35(5.14)(1.2)
\]

\[
\boxed{q_{ult}\approx 216\ \text{kPa}}
\]

Thus, the **ultimate gross bearing pressure** is approximately:

\[
\boxed{216\ \text{kPa}}
\]

The footing area is:

\[
A=\frac{\pi D^2}{4}
=\frac{\pi(2)^2}{4}
=\pi=3.142\ \text{m}^2
\]

Hence, the corresponding ultimate vertical load is:

\[
Q_{ult}=q_{ult}A
=216(3.142)
\]

\[
\boxed{Q_{ult}\

## What to notice

The model picked the bearing capacity factors and did the arithmetic itself. It gave no source
for the factors and no check on the number. Notebooks 7b and 7c close that gap. Retrieval
supplies the source. Tools do the arithmetic.